# Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Student:** Sharmila U.  
**Companies:** Tesla (TSLA) and GameStop (GME)

This notebook retrieves historical stock prices and company revenue data, cleans the revenue tables, displays the requested rows, and plots stock-price and revenue trends.

> Run the cells from top to bottom. The notebook needs an internet connection and the packages `yfinance`, `pandas`, `requests`, `beautifulsoup4`, and `matplotlib`.


## Setup

In [ ]:
# If a package is missing in your environment, uncomment this line and run it:
# %pip install yfinance pandas requests beautifulsoup4 lxml matplotlib

import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Helper functions

The stock-price data comes from Yahoo Finance through `yfinance`. Revenue tables are retrieved from Macrotrends when the website permits automated access. Some websites may change their HTML layout or block requests; if that happens, use the course-provided notebook/data source and keep the same cleaning steps.


In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
}

def get_stock_data(ticker, start="2000-01-01"):
    """Download historical stock data and return a regular DataFrame."""
    data = yf.download(ticker, start=start, progress=False, auto_adjust=False)
    if data.empty:
        raise ValueError(f"No stock data was returned for {ticker}. Check your internet connection.")
    # Recent yfinance versions may return MultiIndex columns.
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    data = data.reset_index()
    return data

def get_revenue_data(company_slug):
    """Try to read a company's quarterly revenue table from Macrotrends."""
    url = f"https://www.macrotrends.net/stocks/charts/{company_slug}/revenue"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    tables = pd.read_html(response.text)
    if not tables:
        raise ValueError("No HTML tables were found on the revenue page.")
    # Choose the table containing a Date and Revenue column.
    for table in tables:
        cols = [str(c).strip().lower() for c in table.columns]
        if any("date" in c for c in cols) and any("revenue" in c for c in cols):
            table.columns = [str(c).strip() for c in table.columns]
            return table
    raise ValueError("Could not identify the revenue table. The website may have changed its layout.")

def clean_revenue(df):
    """Clean date and revenue values; remove dollar signs and commas."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    date_col = next((c for c in df.columns if "date" in c.lower()), df.columns[0])
    revenue_col = next((c for c in df.columns if "revenue" in c.lower()), df.columns[1])
    df = df.rename(columns={date_col: "Date", revenue_col: "Revenue"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Revenue"] = (
        df["Revenue"].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    df["Revenue"] = pd.to_numeric(df["Revenue"], errors="coerce")
    df = df.dropna(subset=["Date", "Revenue"]).sort_values("Date").reset_index(drop=True)
    return df


## Question 1 — Tesla stock data

Retrieve Tesla historical stock data and display the first five rows of `tesla_data`.


In [ ]:
tesla_data = get_stock_data("TSLA")
print("First five rows of tesla_data:")
display(tesla_data.head())


## Question 2 — Tesla revenue data

Retrieve Tesla revenue data, clean it, and display the last five rows of `tesla_revenue`.


In [ ]:
try:
    tesla_revenue = clean_revenue(get_revenue_data("TSLA/tesla"))
    print("Last five rows of tesla_revenue:")
    display(tesla_revenue.tail())
except Exception as error:
    print("Tesla revenue scraping was unsuccessful:", error)
    print("If the site blocks access, run this cell in the course notebook environment or use its supplied revenue source.")


## Question 3 — GameStop stock data

Retrieve GameStop historical stock data and display the first five rows of `gme_data`.


In [ ]:
gme_data = get_stock_data("GME")
print("First five rows of gme_data:")
display(gme_data.head())


## Question 4 — GameStop revenue data

Retrieve GameStop revenue data, clean it, and display the last five rows of `gme_revenue`.


In [ ]:
try:
    gme_revenue = clean_revenue(get_revenue_data("GME/gamestop"))
    print("Last five rows of gme_revenue:")
    display(gme_revenue.tail())
except Exception as error:
    print("GameStop revenue scraping was unsuccessful:", error)
    print("If the site blocks access, run this cell in the course notebook environment or use its supplied revenue source.")


## Plotting functions

These charts show the historical closing stock price and the available revenue observations. The x-axis is formatted as dates.


In [ ]:
def make_stock_plot(stock_df, company_name):
    df = stock_df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df = df.dropna(subset=["Date", "Close"])
    plt.figure(figsize=(12, 5))
    plt.plot(df["Date"], df["Close"])
    plt.title(f"{company_name} Historical Share Price")
    plt.xlabel("Date")
    plt.ylabel("Closing price (USD)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def make_revenue_plot(revenue_df, company_name):
    if revenue_df is None or revenue_df.empty:
        print(f"No revenue data is available to plot for {company_name}.")
        return
    df = revenue_df.copy().dropna(subset=["Date", "Revenue"])
    plt.figure(figsize=(12, 5))
    plt.plot(df["Date"], df["Revenue"], marker="o", markersize=3)
    plt.title(f"{company_name} Quarterly Revenue")
    plt.xlabel("Date")
    plt.ylabel("Revenue (as reported by source)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Question 5 — Tesla graphs

Display both Tesla graphs: historical share price and revenue.


In [ ]:
make_stock_plot(tesla_data, "Tesla")
if "tesla_revenue" in globals() and not tesla_revenue.empty:
    make_revenue_plot(tesla_revenue, "Tesla")


## Question 6 — GameStop graphs

Display both GameStop graphs: historical share price and revenue.


In [ ]:
make_stock_plot(gme_data, "GameStop")
if "gme_revenue" in globals() and not gme_revenue.empty:
    make_revenue_plot(gme_revenue, "GameStop")


## Additional assignment questions — short answers

**Question 2: Which parsers can be used in BeautifulSoup to parse HTML data?**  
BeautifulSoup supports parsers such as `html.parser`, `lxml`, and `html5lib` (if installed).

**Question 3: Why convert the `Date` column to datetime?**  
It allows dates to be sorted and filtered correctly and supports time-series plotting and date-based analysis.

**Question 4: How do you remove `$` and commas from a `Price` column?**  
Use string replacement, for example: `df["Price"] = df["Price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)`.

**Question 5: What is the `Revenue` value in the last row of the extracted `tesla_revenue` dataset?**  
Read the value in the final row after running `display(tesla_revenue.tail())`. It depends on the exact dataset version returned by the source.

**Question 6: What is the `Close` price in the first row of the extracted GameStop stock data (Date: 2002-02-13)?**  
Check the first row of `gme_data` after running the download cell. The exact value can vary with the selected price field and data-source adjustments.

**Question 7: What trend is visible in Tesla's stock price and revenue between 2018 and 2021?**  
Tesla's share price rose substantially during this period. Revenue also generally increased, indicating growth in the business alongside the sharp rise in market valuation.

**Question 8: What key observation can be made about GameStop's stock price and revenue?**  
GameStop's share price experienced a sharp, unusual increase around early 2021, while its revenue did not rise by a comparable amount. This illustrates that stock prices can be influenced by market expectations and investor activity, not only by current revenue.


## Optional: inspect the specific values for Questions 5 and 6

Run this cell to print the last Tesla revenue row and the GameStop row dated 2002-02-13, if present.


In [ ]:
if "tesla_revenue" in globals() and not tesla_revenue.empty:
    print("Last Tesla revenue row:")
    display(tesla_revenue.tail(1))

if "gme_data" in globals() and not gme_data.empty:
    gme_data["Date"] = pd.to_datetime(gme_data["Date"], errors="coerce")
    target = gme_data.loc[gme_data["Date"].dt.strftime("%Y-%m-%d") == "2002-02-13"]
    print("GameStop row for 2002-02-13:")
    display(target[["Date", "Close"]] if not target.empty else target)
